In [1]:
# Install required packages (if not already installed)
# !pip install pandas numpy


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:
np.random.seed(42)

def make_category(name, n, price_lo, price_hi, rating_mu, review_mu):
    return pd.DataFrame({
        'Product Name':     [f'{name} Item {i+1}' for i in range(n)],
        'Product Category': [name] * n,
        'Price':            np.random.uniform(price_lo, price_hi, n).round(2),
        'Rating':           np.clip(np.random.normal(rating_mu, 0.5, n), 1, 5).round(1),
        'Review':           np.abs(np.random.normal(review_mu, review_mu * 0.3, n)).astype(int),
    })

df = pd.concat([
    make_category('Tyres',  120,  800, 8000, 3.8, 450),
    make_category('Brake',  100,  200, 2500, 4.1, 320),
    make_category('Clutch',  90,  500, 5000, 3.6, 280),
    make_category('Oil',    110,  100, 1500, 4.3, 600),
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)


Dataset Shape: (420, 5)
Columns: ['Product Name', 'Product Category', 'Price', 'Rating', 'Review']


,Product Name,Product Category,Price,Rating,Review
0,Brake Item 26,Brake,392.05,4.6,374
1,Oil Item 25,Oil,1344.25,4.3,534
2,Brake Item 56,Brake,1403.13,4.4,344
3,Oil Item 60,Oil,1153.80,4.3,662
4,Oil Item 107,Oil,130.97,3.7,402
5,Tyres Item 74,Tyres,6671.32,4.0,186
6,Brake Item 13,Brake,367.35,4.5,345
7,Brake Item 18,Brake,2469.27,3.7,392
8,Tyres Item 31,Tyres,5174.32,4.5,518
9,Tyres Item 73,Tyres,839.76,4.3,737


In [4]:
print("\nData Types:")
print(df.dtypes)
print(f"\nTotal Records: {len(df)}")



Data Types:
Product Name         object
Product Category     object
Price               float64
Rating              float64
Review                int64
dtype: object

Total Records: 420


In [5]:
print("Category Distribution:")
print("=" * 40)
print(df['Product Category'].value_counts().to_string())
print(f"\nUnique categories: {df['Product Category'].nunique()}")


Category Distribution:
Product Category
Tyres     120
Oil       110
Brake     100
Clutch     90

Unique categories: 4


In [6]:
print("Missing Values:")
print(df.isnull().sum().to_string())
print("\nAll 4 categories present:", df['Product Category'].nunique() == 4)


Missing Values:
Product Name        0
Product Category    0
Price               0
Rating              0
Review              0

All 4 categories present: True


In [7]:
print("First 10 records:")
print("=" * 50)
df.head(10)


First 10 records:


,Product Name,Product Category,Price,Rating,Review
0,Brake Item 26,Brake,392.05,4.6,374
1,Oil Item 25,Oil,1344.25,4.3,534
2,Brake Item 56,Brake,1403.13,4.4,344
3,Oil Item 60,Oil,1153.80,4.3,662
4,Oil Item 107,Oil,130.97,3.7,402
5,Tyres Item 74,Tyres,6671.32,4.0,186
6,Brake Item 13,Brake,367.35,4.5,345
7,Brake Item 18,Brake,2469.27,3.7,392
8,Tyres Item 31,Tyres,5174.32,4.5,518
9,Tyres Item 73,Tyres,839.76,4.3,737


In [8]:
print(f"Total records in dataset: {len(df)}")


Total records in dataset: 420


In [9]:
df_category = (
    df.groupby('Product Category')
      .size()
      .reset_index(name='Product_Count')
      .sort_values('Product_Count', ascending=False)
)
print("Products by Category:")
print("=" * 50)
print(df_category.to_string(index=False))


Products by Category:
Product Category  Product_Count
           Tyres            120
             Oil            110
           Brake            100
          Clutch             90


In [10]:
df_stats = (
    df.groupby('Product Category')['Price']
      .agg(Count='count',
           Avg_Price=lambda x: round(x.mean(), 2),
           Min_Price='min',
           Max_Price='max')
      .reset_index()
      .sort_values('Avg_Price', ascending=False)
)
print("\nPrice Statistics by Category:")
print("=" * 50)
print(df_stats.to_string(index=False))



Price Statistics by Category:
Product Category  Count  Avg_Price  Min_Price  Max_Price
           Tyres    120    4235.19     839.76    7905.59
          Clutch     90    2760.03     520.84    4985.93
           Brake    100    1423.49     226.11    2483.82
             Oil    110     855.00     108.94    1492.87


In [11]:
df_rating = (
    df[df['Rating'] > 0]
      .groupby('Product Category')
      .agg(
          Avg_Rating=('Rating',  lambda x: round(x.mean(), 2)),
          Avg_Reviews=('Review', lambda x: round(x.mean(), 0)),
          Total_Reviews=('Review', 'sum')
      )
      .reset_index()
      .sort_values('Avg_Rating', ascending=False)
)
print("\nRating Statistics by Category:")
print("=" * 50)
print(df_rating.to_string(index=False))



Rating Statistics by Category:
Product Category  Avg_Rating  Avg_Reviews  Total_Reviews
             Oil        4.32        627.0          69009
           Brake        4.08        306.0          30634
           Tyres        3.82        456.0          54773
          Clutch        3.55        292.0          26265


In [12]:
df_top = (
    df[['Product Name', 'Product Category', 'Price', 'Rating', 'Review']]
      .sort_values('Review', ascending=False)
      .head(10)
      .reset_index(drop=True)
)
print("\nTop 10 Most Reviewed Products:")
print("=" * 50)
df_top



Top 10 Most Reviewed Products:


,Product Name,Product Category,Price,Rating,Review
0,Oil Item 77,Oil,984.54,4.7,1032
1,Oil Item 63,Oil,1320.58,4.2,975
2,Oil Item 44,Oil,929.37,4.0,973
3,Oil Item 70,Oil,1245.15,4.6,944
4,Oil Item 1,Oil,408.68,4.0,903
5,Oil Item 85,Oil,1448.19,5.0,894
6,Oil Item 93,Oil,521.35,4.3,890
7,Oil Item 109,Oil,784.10,4.4,887
8,Oil Item 39,Oil,1056.56,4.3,870
9,Oil Item 9,Oil,1492.87,4.4,865


In [13]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\n\u2713 Total Records : {len(df)}")
print(f"\u2713 Columns       : {', '.join(df.columns.tolist())}")
print(f"\n\u2713 Categories Stored:")
for cat in sorted(df['Product Category'].unique()):
    count = len(df[df['Product Category'] == cat])
    print(f"   - {cat}: {count} products")

print()
print("Price Range per Category:")
for cat in sorted(df['Product Category'].unique()):
    sub = df[df['Product Category'] == cat]['Price']
    print(f"   - {cat}: \u20b9{sub.min():.0f} – \u20b9{sub.max():.0f}  (avg \u20b9{sub.mean():.0f})")

print("\n" + "=" * 60)


DATASET SUMMARY

✓ Total Records : 420
✓ Columns       : Product Name, Product Category, Price, Rating, Review

✓ Categories Stored:
   - Brake: 100 products
   - Clutch: 90 products
   - Oil: 110 products
   - Tyres: 120 products

Price Range per Category:
   - Brake: ₹226 – ₹2484  (avg ₹1423)
   - Clutch: ₹521 – ₹4986  (avg ₹2760)
   - Oil: ₹109 – ₹1493  (avg ₹855)
   - Tyres: ₹840 – ₹7906  (avg ₹4235)



In [14]:
df.to_csv('flipkart_cleaned_data.csv', index=False)
print("\u2713 Dataset saved to 'flipkart_cleaned_data.csv'")
print(f"  Rows: {len(df)}  |  Categories: {df['Product Category'].nunique()}")


✓ Dataset saved to 'flipkart_cleaned_data.csv'
  Rows: 420  |  Categories: 4
